In [1]:
#processing the text file
import os

def write_fasta_file(output_file, sequence_list):
    with open(output_file, "w") as f:
        for i, sequence in enumerate(sequence_list, start=1):
            f.write(f">Sequence_{i}\n{sequence}\n")

# Read the input text file
input_file = "Main/amp_val_neg.txt"
with open(input_file, "r") as f:
    lines = f.readlines()

# Remove leading/trailing whitespace and empty lines
sequences = [line.strip() for line in lines if line.strip()]

# Get the base name of the input file (without extension)
base_name = os.path.splitext(os.path.basename(input_file))[0]

# Define the output FASTA file name
output_fasta_file = f"{base_name}.fasta"

# Specify the output directory for the FASTA file
output_directory = "Main/"

# Construct the full path for the output FASTA file
output_file_path = os.path.join(output_directory, output_fasta_file)

# Write the sequences to the output FASTA file
write_fasta_file(output_file_path, sequences)

print("FASTA file created successfully at:", output_file_path)

FASTA file created successfully at: Main/amp_val_neg.fasta


In [2]:
#this involves the process of data creation
from Bio import SeqIO
from Bio.SeqUtils.ProtParam import ProteinAnalysis as PA
from modlamp.descriptors import PeptideDescriptor, GlobalDescriptor
from sklearn.model_selection import train_test_split,KFold
import pandas as pd
import os, re, math, platform
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from numpy import linalg as la
import os
import random
import warnings
warnings.filterwarnings('ignore')
import random
from fea_extract import read_fasta,insert_AAC,insert_DPC,insert_CKSAAGP,insert_CTD,insert_PAAC,insert_AAI,insert_GTPC,insert_QSO,insert_AAE,insert_PSAAC,insert_word2int,insert_ASDC
from tools import del_data
seed = 10
from pathlib import Path
Path('./results/process_data/').mkdir(exist_ok=True,parents=True)
Path('./results/balance/').mkdir(exist_ok=True,parents=True)

In [3]:
#fasta to csv format conversion
def del_data(inFile):
    seq = read_fasta(inFile)
    seqname = seq.to_numpy()
    newseq = []
    j = seqname.shape[0]
    for i in range(j):
        if 6 <= len(seqname[i][1])<=100:
            newseq.append(seqname[i])
    newseq = np.array(newseq)
    print('New Shape：', newseq.shape)
    newseq = pd.DataFrame(data=newseq, columns=["Id", "Sequence"])
    return newseq

In [4]:
from Bio import SeqIO

# Read the first FASTA file
fasta_file1 = "Main/amp_train_neg.fasta"
sequences1 = list(SeqIO.parse(fasta_file1, "fasta"))

# Read the second FASTA file
fasta_file2 = "Main/amp_val_neg.fasta"
sequences2 = list(SeqIO.parse(fasta_file2, "fasta"))

# Combine the sequences from both files
combined_sequences = sequences1 + sequences2

# Specify the output FASTA file name
output_fasta_file = "AMP_TRAIN.fasta"

# Write the combined sequences to the output FASTA file
with open(output_fasta_file, "w") as f:
    SeqIO.write(combined_sequences, f, "fasta")

print("Combined FASTA file created successfully.")

Combined FASTA file created successfully.


In [5]:
seq_ACP = del_data('ACP_TRAIN.fasta')
seq_AMP = del_data('AMP_TRAIN.fasta')

New Shape： (845, 2)
New Shape： (859, 2)


In [6]:
from tools import pro_data

In [7]:
#80% AND 20% SPLIT
prefix = [seq_ACP,seq_AMP]
Prefix = ['Seq_ACP','Seq_AMP']
for i in range(2):
    df_train,df_test = train_test_split(prefix[i],random_state=seed,test_size=.2)
    df_train.to_csv(os.path.join( "data/{:s}_train.csv".format(Prefix[i])), index=False)
    df_test.to_csv(os.path.join("data/{:s}_test.csv".format(Prefix[i])), index=False)
    print("Done!")

Done!
Done!


In [8]:
#PREPARIONG THE TRAIN AND TEST DATA FOR TRAINING
train_sets = {
    lab:pd.read_csv('data/{:s}_train.csv'.format(lab))
    for lab in ['Seq_ACP','Seq_AMP']
}
test_sets = {
    lab:pd.read_csv('data/{:s}_test.csv'.format(lab))
    for lab in ['Seq_ACP','Seq_AMP']
}
train_sets['Seq_ACP'].loc[:, 'Label'] = 1 
train_sets['Seq_AMP'].loc[:, 'Label'] = 0 
test_sets['Seq_ACP'].loc[:, 'Label'] = 1
test_sets['Seq_AMP'].loc[:, 'Label'] = 0


all_train = pd.concat([train_sets['Seq_ACP'],train_sets['Seq_AMP']],ignore_index='ignore')
all_test = pd.concat([test_sets['Seq_ACP'],test_sets['Seq_AMP']],ignore_index='ignore')
X_train = all_train.iloc[:,0:2]
X_test = all_test.iloc[:,0:2]
y_train = all_train['Label']
y_test = all_test["Label"]
X_test.to_csv('data/test/X_test.csv',index = False)
y_test.to_csv('data/test/y_test.csv',index = False)
X_train.to_csv('data/train/X_train.csv',index = False)
y_train.to_csv('data/train/y_train.csv',index = False)

In [9]:
X_train

,Id,Sequence
0,Sequence_194,GIGKFLKKAKKFGKAFVKILKK
1,Sequence_581,GIPCAESCVWIPCTVTALVGCSCSDKVCYN
2,Sequence_310,GLLSVLGSVAQHVLPHVVPVIAEHL
3,Sequence_177,VAKLLAKLAKKLL
4,Sequence_632,GASCGETCFTGICFTAGCSCNPWPTCTRN
...,...,...
1358,Sequence_370,RILSILRHQNLLKELQDLALQGAK
1359,Sequence_321,QKIAEKFSGTRRG
1360,Sequence_528,GIFGKILGVGKKVLCGLSGVC
1361,Sequence_126,GLGKAQCAALWLQCASGGTIGCGGGAVACQNYRQFCR


In [10]:
y_train

0       1
1       1
2       1
3       1
4       1
       ..
1358    0
1359    0
1360    0
1361    0
1362    0
Name: Label, Length: 1363, dtype: int64

In [11]:
#encdoing the peptide sequeneces
insert_list=[insert_AAC,insert_DPC,insert_CKSAAGP,insert_PSAAC,insert_CTD,insert_GTPC,
             insert_QSO,insert_AAE,insert_AAI,insert_ASDC,insert_PAAC,pro_data]
insert_str=["insert_AAC","insert_DPC","insert_CKSAAGP","insert_PSAAC","insert_CTD",
            "insert_GTPC","insert_QSO","insert_AAE","insert_AAI","insert_ASDC","insert_PAAC",'insert_All_data']

In [12]:
n = len(insert_list)
for j in range(n):
    print("encoding{}_{}".format(str(j),insert_str[j][7:]))
    X_test = pd.read_csv('data/test/X_test.csv')
    X_train = pd.read_csv('data/train/X_train.csv')
    df_seq_train = insert_list[j](X_train)
    df_seq_train.to_csv('results/process_data/train_{}.csv'.format(insert_str[j][7:]),index =False )
    df_seq_test = insert_list[j](X_test)
    df_seq_test.to_csv('results/process_data/test_{}.csv'.format(insert_str[j][7:]),index =False )

encoding0_AAC
encoding1_DPC
encoding2_CKSAAGP
encoding3_PSAAC
encoding4_CTD
encoding5_GTPC
encoding6_QSO
encoding7_AAE
encoding8_AAI
encoding9_ASDC
encoding10_PAAC
encoding11_All_data


In [13]:
#wordtoint encoding for peptides
from tools import padseq

In [14]:
X_test = pd.read_csv('data/test/X_test.csv')
X_train = pd.read_csv('data/train/X_train.csv')
new_seq_train = padseq(X_train)
new_seq_test = padseq(X_test)
word2_seq_train = insert_word2int(new_seq_train)
word2_seq_train.to_csv('results/process_data/train_word2int.csv',index=False)

word2_seq_test = insert_word2int(new_seq_test)
word2_seq_test.to_csv('results/process_data/test_word2int.csv',index=False)

In [15]:
#encoding a balanced dataset
insert_list=[insert_AAC,insert_DPC,insert_CKSAAGP,insert_PSAAC,insert_CTD,insert_GTPC,
             insert_QSO,insert_AAE,insert_AAI,insert_ASDC,insert_PAAC,pro_data]
insert_str=["insert_AAC","insert_DPC","insert_CKSAAGP","insert_PSAAC","insert_CTD",
            "insert_GTPC","insert_QSO","insert_AAE","insert_AAI","insert_ASDC","insert_PAAC","pro_data"]